In [ ]:
__author__ = "Alireza Sadabadi"
__copyright__ = "Copyright (c) 2026 Alireza Sadabadi. All rights reserved."
__credits__ = ["Alireza Sadabadi"]
__license__ = "Apache"
__version__ = "2.0"
__maintainer__ = "Alireza Sadabadi"
__email__ = "alirezasadabady@gmail.com"
__status__ = "Test"
__doc__ = "you can see the tutorials in https://youtube.com/@alirezasadabadi?si=d8o7LK_Ai1Hf68is"

import pandas as pd
import numpy as np
import pandas_ta as pdta
from Meta import *

if not mt5.initialize():
    print("initialize() failed, error code =",mt5.last_error())
    quit()

df = Meta.GetRates(symbol='XAUUSD', number_of_data=10000, timeFrame=mt5.TIMEFRAME_M1)
spikeCandleSize = 1.5 # the spike candle size must be atleast x times bigger than around candles
pGapSize = 1 # distance from low of candle after spike and high of candle before spike
backCandleSampleCount = 3
for lag in range(1 , backCandleSampleCount + 1):
    colLow = "low-{}".format(lag)
    colHigh = "high-{}".format(lag)
    colClose = "close-{}".format(lag)
    colOpen = "open-{}".format(lag)
    df[colLow] = df.low.shift(lag)
    df[colHigh] = df.high.shift(lag)
    df[colClose] = df.close.shift(lag)
    df[colOpen] = df.open.shift(lag)
df.dropna(inplace = True)
df.loc[:,['close','close-1','close-2','low-1','low-3']]

In [17]:
buy0 = df['low'] < df['low-1']

buy1 = df['close-1'] > df['close-2']
buy2 = df['open-1'] > df['open-2']
buy3 = df['close-2'] > df['close-3']
buy4 = df['open-2'] > df['open-3']
buy5 = df['close-1'] > df['open-1']
buy6 = df['close-2'] > df['open-2']
buy7 = df['close-3'] > df['open-3']

pGapBuy = df['low-1'] > df['high-3'] + pGapSize
spikeBuy = (df['close-2'] - df['open-2'] > spikeCandleSize * (df['close-1']-df['open-1'])) & (df['close-2'] - df['open-2'] > spikeCandleSize * (df['close-3']-df['open-3'])) & (df['close-2'] - df['open-2'] > spikeCandleSize * (df['close']-df['open']))

buy = buy0 & buy1 & buy2 & buy3 & buy4 & buy5 & buy6 & buy7 & pGapBuy & spikeBuy

sell0 = df['high'] > df['high-1']

sell1 = df['close-1'] < df['close-2']
sell2 = df['open-1'] < df['open-2']
sell3 = df['close-2'] < df['close-3']
sell4 = df['open-2'] < df['open-3']
sell5 = df['close-1'] < df['open-1']
sell6 = df['close-2'] < df['open-2']
sell7 = df['close-3'] < df['open-3']

pGapSell = df['high-1'] < df['low-3'] - pGapSize
spikeSell = (df['open-2'] - df['close-2'] > spikeCandleSize * (df['open-1']-df['close-1'])) & (df['open-2'] - df['close-2'] > spikeCandleSize * (df['open-3']-df['close-3'])) & (df['open-2'] - df['close-2'] > spikeCandleSize * (df['open']-df['close']))

sell = sell0 & sell1 & sell2 & sell3 & sell4 & sell5 & sell6 & sell7 & pGapSell & spikeSell

df["position"]= np.where(buy, 1, np.nan)
df["position"]= np.where(sell, -1, df.position)

In [ ]:
df[df.position == 1].count()
#df.to_csv('temp.csv')

In [ ]:
df = df.loc[:,["open", "high", "low", "low-1", "tick_volume", "position", "low-3", "high-3", "low-1", "high-1"]]
df.reset_index(inplace=True)
df.columns = ['Local time', 'Open', 'High', 'Low', 'Close', 'Volume', 'signal', 'sl_buy', 'sl_sell', 'entry_buy', 'entry_sell']
df.index = pd.DatetimeIndex(df['Local time'])

def Signal():
    return df.signal

from backtesting import Strategy

class MyStrategy(Strategy): 
    def init(self):
        super().init()
        self.signal1 = self.I(Signal)

    def next(self):
        super().next() 
        
        # buy                        
        if self.signal1[-1]==1 and len(self.trades) == 0:
            
            sl = self.data['sl_buy'][-1]
            tp = (self.data['entry_buy'][-1] - sl) + self.data['entry_buy'][-1]
            #self.buy(stop=self.data['entry_buy'][-1], sl=sl, tp=tp, size=0.1)
            self.buy(sl=sl, tp=tp, size=0.1)
            
        elif self.signal1[-1]==-1 and len(self.trades) == 0:
            
            sl = self.data['sl_sell'][-1]
            tp = self.data['entry_sell'][-1] - (sl - self.data['entry_sell'][-1])
            self.sell(stop=self.data['entry_sell'][-1], sl=sl, tp=tp, size=0.1)

In [19]:
df = df.loc[:,["open", "high", "low", "close", "tick_volume", "position", "low-3", "high-3", "low-1", "high-1"]]
df.reset_index(inplace=True)
df.columns = ['Local time', 'Open', 'High', 'Low', 'Close', 'Volume', 'signal', 'sl_buy', 'sl_sell', 'entry_buy', 'entry_sell']
df.index = pd.DatetimeIndex(df['Local time'])

from backtesting import Strategy

def Signal():
    return df.signal


class MyStrategy(Strategy):

    def init(self):
        super().init()
        self.signal1 = self.I(Signal)

    def next(self):
        super().next()

        # -------------------------------
        #  افزودن حد ضرر و سود به معامله‌ای که تازه باز شده
        # -------------------------------
        if self.trades:
            trade = self.trades[-1]

            # معامله باز است و هنوز حد ضرر و سود ندارد
            if trade.exit_price is None and trade.sl is None:

                if trade.is_long:
                    sl = self.data.sl_buy[-1]
                    entry = trade.entry_price
                    tp = (entry - sl) + entry

                else:  # short
                    sl = self.data.sl_sell[-1]
                    entry = trade.entry_price
                    tp = entry - (sl - entry)

                trade.sl = sl
                trade.tp = tp

        # -------------------------------
        # منطق ورود بدون تعیین حد ضرر و سود
        # -------------------------------
        if len(self.trades) == 0:

            if self.signal1[-1] == 1:
                self.buy(stop=self.data.entry_buy[-1], size=0.1)

            elif self.signal1[-1] == -1:
                self.sell(stop=self.data.entry_sell[-1], size=0.1)


In [20]:
from backtesting import Backtest

backtest = Backtest(df, MyStrategy, cash=10000, commission=0.0001, margin = 1/5)
result = backtest.run()
result

Start                     2025-12-17 03:49:00
End                       2025-12-29 13:45:00
Duration                     12 days 09:56:00
Exposure Time [%]                     5.47164
Equity Final [$]                  10012.53735
Equity Peak [$]                   10037.75797
Return [%]                            0.12537
Buy & Hold Return [%]                 2.99696
Return (Ann.) [%]                     4.12752
Volatility (Ann.) [%]                 2.66327
CAGR [%]                              2.57609
Sharpe Ratio                          1.54979
Sortino Ratio                         2.63239
Calmar Ratio                         16.42743
Max. Drawdown [%]                    -0.25126
Avg. Drawdown [%]                    -0.04829
Max. Drawdown Duration        3 days 11:58:00
Avg. Drawdown Duration        0 days 11:53:00
# Trades                                   57
Win Rate [%]                         50.87719
Best Trade [%]                         0.2446
Worst Trade [%]                   